# A worked example of L2Boosting

This example traces five iterations of a six-observation boosting fit by hand
using the values returned by the engine. It applies the algorithm in
[](functional_gradient_boosting.ipynb) before deriving the learner structures
in [](base_learners.ipynb).

```{admonition} Where this example reappears
:class: tip
The same three iterations are traced through the Rust engine in
[](../implementation/engine) (which buffers hold what, where the back-solve
happens) and run at scale in [](../usage/quickstart) (400 observations, the
same two-learner structure).
```

## The problem

Six observations, a Gaussian response, and two candidate linear base learners.
Only `x1` carries signal; `x2` is noise. The example includes an offset,
competing learner fits, a selected learner, and an update.

| $i$ | 1 | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|---|
| $y_i$        | 2.0 | 0.5 | 1.5 | 3.0 | 1.0 | 2.5 |
| $x_{1,i}$    | 1.0 | 0.0 | 1.0 | 2.0 | 0.0 | 1.0 |
| $x_{2,i}$    | 0.0 | 1.0 | 0.0 | 1.0 | 2.0 | 1.0 |

We fit `y ~ bols(x1) + bols(x2)` with `Gaussian()` loss, step length
$\nu = 0.1$, for $m_{\text{stop}} = 5$ iterations. The base learners use the
default `df = 1`, which for a single centered column is plain OLS (a ridge
penalty with `df = 1` collapses to $\lambda = 0$).

## Step 0 — the offset

The offset is the constant predictor that minimizes the risk. For Gaussian
loss that is the weighted mean of $y$:

$$
\hat f^{[0]} \;=\; \frac{\sum_i w_i y_i}{\sum_i w_i} \;=\; \frac{2.0+0.5+1.5+3.0+1.0+2.5}{6} \;=\; 1.75.
$$

Every observation starts at the same value: $f^{[0]} = (1.75, 1.75, \dots, 1.75)$.
The risk at the offset is the variance scaled by $n$:

$$
\mathcal R(f^{[0]}) \;=\; \tfrac{1}{2}\sum_i (y_i - 1.75)^2 \;=\; 4.375.
$$

This is `risk[0]` in the path.

## Step 1 — the first boosting iteration

### (a) The negative gradient (pseudo-residual)

For Gaussian loss the negative gradient is the residual
$u_i = -\partial\rho/\partial f = y_i - f_i$:

| $i$ | 1 | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|---|
| $u_i = y_i - 1.75$ | $+0.25$ | $-1.25$ | $-0.25$ | $+1.25$ | $-0.75$ | $+0.75$ |

### (b) Fit each base learner to $u$

Each `bols(x)` centers its column and fits it to $u$ by least squares. With
$x_1$ centered, $\tilde x_1 = x_1 - \bar x_1$ where $\bar x_1 = 5/6 \approx 0.833$:

$$
\tilde x_1 = (0.167,\; -0.833,\; 0.167,\; 1.167,\; -0.833,\; 0.167).
$$

The OLS coefficient and its SSE against $u$ are

$$
\hat\beta_1 = \frac{\tilde x_1^\top u}{\tilde x_1^\top \tilde x_1} = \frac{2.4583}{2.1389} = 1.147, \qquad
\mathrm{SSE}_1 = \sum_i (u_i - \hat\beta_1 \tilde x_{1,i})^2 = 0.647.
$$

Repeating for `x2` (centered mean $\bar x_2 = 5/6$ as well):

$$
\hat\beta_2 = -0.265, \qquad \mathrm{SSE}_2 = 4.177.
$$

### (c) Select the best component

The engine selects the learner with the **smallest weighted SSE** against the
pseudo-residual:

| Learner | $\hat\beta$ | SSE against $u$ |
|---|---|---|
| `bols(x1)` | $+1.147$ | **0.647** $\leftarrow$ winner |
| `bols(x2)` | $-0.265$ | 4.177 |

`x1` explains the residual far better than `x2`. **Select $j^\star = 1$.**

### (d) Update the predictor

Add a fraction $\nu = 0.1$ of the winner's fit:

$$
f^{[1]} \;=\; f^{[0]} + \nu \cdot \hat\beta_1 \tilde x_1 \;=\; 1.75 + 0.1 \cdot 1.147 \cdot \tilde x_1.
$$

The new risk is $\mathcal R(f^{[1]}) = 3.667$ (from $4.375$ at the offset).

```{admonition} Why `x2` was not selected
:class: note
The residual $u$ points in the direction of $x_1$ (large where $x_1$ is large,
small where $x_1$ is small). The `x2` column is nearly orthogonal to $u$, so
its fit is weak and its SSE stays near $\sum u_i^2$.
The learner that most reduces the risk is the one whose
direction aligns with the residual. A learner that never aligns never wins, and
its coefficient stays exactly zero — automatic variable selection.
```

## Steps 2–5 — the rest of the path

Each subsequent iteration repeats the same four sub-steps on the new residual.
Because `x1` keeps winning, the predictor accumulates more and more of `x1`'s
direction. The numbers:

| $m$ | $\hat\beta_1^{[m]}$ | $\mathrm{SSE}_1$ | $\hat\beta_2^{[m]}$ | $\mathrm{SSE}_2$ | winner | $\mathcal R(f^{[m]})$ |
|---|---|---|---|---|---|---|
| 0 | — | — | — | — | (offset) | 4.375 |
| 1 | 1.147 | 0.647 | $-0.265$ | 4.177 | x1 | 3.667 |
| 2 | 1.032 | 0.647 | $-0.218$ | 3.533 | x1 | 3.093 |
| 3 | 0.929 | 0.647 | $-0.175$ | 3.006 | x1 | 2.628 |
| 4 | 0.836 | 0.647 | $-0.137$ | 2.575 | x1 | 2.252 |
| 5 | 0.753 | 0.647 | $-0.102$ | 2.222 | x1 | 1.947 |

The per-step $\hat\beta_1^{[m]}$ shrinks each iteration because the residual
shrinks (the unfitted part gets smaller), but its SSE against the shrinking
residual stays at $0.647$ — specific to this design; $\mathrm{SSE}_1 < \mathrm{SSE}_2$
holds at every step.

## The fitted model after 5 iterations

Summing the damped increments, the aggregated coefficient on `x1` (centered
scale) is

$$
\hat\beta_1^{\text{agg}} = \nu \sum_{m=1}^{5} \hat\beta_1^{[m]} = 0.1 \cdot (1.147 + 1.032 + 0.929 + 0.836 + 0.753) = 0.470.
$$

The final predictor, evaluated at each observation:

| $i$ | 1 | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|---|
| $\hat f_i^{[5]}$ | 1.828 | 1.359 | 1.828 | 2.298 | 1.359 | 1.828 |

`x2` was never selected in these five iterations, so its coefficient is exactly
$0$. Its contribution remains zero until an iteration selects it.

## Verification against the engine

Running the same fit through `mboostrs` reproduces these numbers to four
decimals:

In [1]:
import numpy as np
import mboostrs as mb

y  = np.array([2.0, 0.5, 1.5, 3.0, 1.0, 2.5])
x1 = np.array([1.0, 0.0, 1.0, 2.0, 0.0, 1.0])
x2 = np.array([0.0, 1.0, 0.0, 1.0, 2.0, 1.0])

fit = mb.glmboost(
    "y ~ bols(x1) + bols(x2)",
    data={"y": y, "x1": x1, "x2": x2},
    family=mb.Gaussian(),
    control=mb.boost_control(mstop=5, nu=0.1),
)
print("offset   :", fit.offset_)
print("selected :", fit.selected)
print("risk     :", np.round(fit.risk_, 4))
print("fitted   :", np.round(fit.fitted_, 4))
print("coef     :", {k: round(v, 4) for k, v in fit.coef().items()})

# Assert the hand-computed numbers against the engine: a change to the
# risk/gradient/selection logic that breaks this chapter's arithmetic fails
# the assertions below.
hand_risk = [4.375, 3.667, 3.093, 2.628, 2.252, 1.947]
hand_fitted = [1.828, 1.359, 1.828, 2.298, 1.359, 1.828]
assert fit.offset_ == 1.75
assert fit.selected == ["x1"] * 5
np.testing.assert_allclose(fit.risk_, hand_risk, atol=5e-4)
np.testing.assert_allclose(fit.fitted_, hand_fitted, atol=5e-4)
np.testing.assert_allclose(fit.coef()["x1"], 0.470, atol=5e-4)

# The two identities in the paragraph below, which were outside this safety
# net and were both wrong: the intercept is the centering correction alone,
# and a prediction is offset + X\u03b2 rather than X\u03b2.
coefs = fit.coef()
np.testing.assert_allclose(
    coefs["(Intercept)"], -coefs["x1"] * x1.mean(), atol=5e-4
)
np.testing.assert_allclose(
    fit.offset_ + coefs["(Intercept)"] + coefs["x1"] * x1, fit.fitted_, atol=1e-12
)
print("\nhand-computed trace matches the engine to 3 decimals.")

offset   : 1.75
selected : ['x1', 'x1', 'x1', 'x1', 'x1']
risk     : [4.375  3.6667 3.093  2.6282 2.2518 1.9469]
fitted   : [1.8283 1.3586 1.8283 2.298  1.3586 1.8283]
coef     : {'(Intercept)': -0.3914, 'x1': 0.4697}

hand-computed trace matches the engine to 3 decimals.


The `coef()` output reports coefficients on the **uncentered** scale, so the
intercept carries the centering correction and nothing else:

$$(\text{Intercept}) = -\hat\beta_1 \cdot \bar x_1 = -0.470 \times 0.8333 = -0.391.$$

The offset is *not* folded in. A prediction reconstructs as
$\hat f = \text{offset} + X\beta$, so at $x_1 = 1$:

$$\hat f = 1.75 + (-0.391) + 0.470 \times 1 = 1.828,$$

matching the fitted table. Dropping the offset gives $-0.391 + 0.470 = 0.079$,
which is short by the offset. A prediction reconstructed from `coef()` must
include the offset.
The cell above asserts both identities.

## Smooth effects with `bbs`

A nonlinear response requires a learner that can represent its shape. Consider
the following hump-shaped response, fitted first with linear learners and then
with a P-spline:

| $i$ | 1 | 2 | 3 | 4 | 5 | 6 | 7 |
|---|---|---|---|---|---|---|---|
| $y_i$        | 0.0 | 0.6 | 0.9 | 1.0 | 0.8 | 0.5 | 0.2 |
| $x_{1,i}$    | 0.0 | 0.5 | 1.0 | 1.5 | 2.0 | 2.5 | 3.0 |
| $x_{2,i}$    | 1.0 | 0.0 | 1.0 | 0.0 | 1.0 | 0.0 | 1.0 |

Fitting `y ~ bols(x1) + bols(x2)` — linear learners for both — produces a poor
fit because a straight line cannot follow the hump. The noise column `x2` is
selected first because its alternation correlates with the residual from a
linear fit, and the risk at $m_{\text{stop}} = 50$ is stuck at
$0.7245$ — barely below the offset risk of $0.8143$.

Replace `bols(x1)` with `bbs(x1, df=5)` — a B-spline basis of degree 3 with
five effective degrees of freedom. `bbs` expands $x_1$ into a set of local
basis functions that together can represent any smooth shape:

```{figure} ../figures/fig-bspline-basis.svg
:width: 75%
:alt: B-spline basis functions
:align: center
The B-spline basis for $x_1$ with `knots = 3, degree = 3` (the default). Each
colored curve is one basis function — non-zero only over a short interval. The
penalty matrix $K$ (a second-order difference penalty on adjacent coefficients)
encourages neighboring basis functions to have similar weights, enforcing
smoothness.
```

A `bbs(x, df=5)` learner uses `df` as its target degrees of freedom. The
internal penalty $\lambda$ is solved so that $\operatorname{tr}(H_\lambda) = 5$,
where $H_\lambda$ is the hat matrix of the penalized fit. Every iteration, this
learner fits its basis to the pseudo-residual under the smoothness constraint,
then competition and selection proceed exactly as in the linear case.

Here is the same hump data with `y ~ bbs(x1, df=5) + bols(x2)`:

In [2]:
import numpy as np
import mboostrs as mb

y  = np.array([0.0, 0.6, 0.9, 1.0, 0.8, 0.5, 0.2])
x1 = np.array([0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0])
x2 = np.array([1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0])

fit = mb.gamboost(
    "y ~ bbs(x1, df=5) + bols(x2)",
    data={"y": y, "x1": x1, "x2": x2},
    family=mb.Gaussian(),
    control=mb.boost_control(mstop=50, nu=0.1),
)
print("offset        :", round(fit.offset_, 4))
print("risk[0] (offset):", round(fit.risk_[0], 4))
print("risk[5]        :", round(fit.risk_[5], 4))
print("risk[50]       :", round(fit.risk_[-1], 4))
print("selected[:8]   :", fit.selected[:8])
print("fitted         :", np.round(fit.fitted_, 4))

assert fit.selected[:8] == ["x1"] * 8, "bbs(x1) should win every early iteration"
np.testing.assert_allclose(fit.risk_[0], 0.8143, atol=5e-4)
np.testing.assert_allclose(fit.risk_[-1], 0.0005, atol=5e-4)
np.testing.assert_allclose(
    fit.fitted_, [0.01, 0.59, 0.91, 0.99, 0.80, 0.51, 0.20], atol=2e-2
)

offset        : 0.5714
risk[0] (offset): 0.8143
risk[5]        : 0.304
risk[50]       : 0.0005
selected[:8]   : ['x1', 'x1', 'x1', 'x1', 'x1', 'x1', 'x1', 'x1']
fitted         : [0.0097 0.5877 0.9088 0.9886 0.7991 0.5064 0.1996]


* **Selection**: `x1` wins all 50 iterations. The `bbs` learner fits the
  hump, while `bols(x2)` never reduces the residual enough to be selected first.
* **Risk**: $0.8143 \to 0.0005$ — nearly zero after 50 iterations.
  The smooth fit tracks the hump almost exactly.
* **Fitted values**: `[0.01, 0.59, 0.91, 0.99, 0.80, 0.51, 0.20]` — within
  0.02 of the true $y$ at every observation.

The per-step risk trace for the first five iterations:

In [3]:
print("Risk path (iterations 0–5):", np.round(fit.risk_[:6], 4))
print("Selected in order         :", fit.selected[:5])

Risk path (iterations 0–5): [0.8143 0.6684 0.5487 0.4505 0.37   0.304 ]
Selected in order         : ['x1', 'x1', 'x1', 'x1', 'x1']


Compare with the linear-only result (run the cell below to confirm):

In [4]:
fit_lin = mb.glmboost(
    "y ~ bols(x1) + bols(x2)",
    data={"y": y, "x1": x1, "x2": x2},
    family=mb.Gaussian(),
    control=mb.boost_control(mstop=50, nu=0.1),
)
print("bols risk[50]:", round(fit_lin.risk_[-1], 4))
print("bols fitted :", np.round(fit_lin.fitted_, 4))
print("bols selected (first 5):", fit_lin.selected[:5])

assert fit_lin.selected[:5] == ["x2"] * 5, "the noise column wins early when the true shape is nonlinear"
np.testing.assert_allclose(fit_lin.risk_[-1], 0.7245, atol=5e-4)

bols risk[50]: 0.7245
bols fitted : [0.4512 0.6782 0.4691 0.696  0.4869 0.7139 0.5048]
bols selected (first 5): ['x2', 'x2', 'x2', 'x2', 'x2']


The linear fit barely improves from the offset, and — because a straight line
is the wrong shape — it picks up the noise column `x2` before `x1`. Boosting
needs flexible base learners: `bols` captures linear trends, `bbs` captures
smooth curves, and the selection contest at each iteration picks whichever
one best matches the current residual.

## Summary

1. **The loss enters only through the pseudo-residual.** Every formula in
   step 1 used $u = y - f$ because the loss was Gaussian. Swap to Binomial and
   $u$ becomes $y - p$; swap to Quantile and $u$ becomes $\tau - \mathbf{1}\{y<f\}$.
   The base learners, the selection contest, and the update are identical.
   One engine serves every family; see [](families.ipynb).

2. **Early stopping limits selected effects.** The linear example selected
   `x1` at every iteration and left `x2` at zero. With $m_{\text{stop}} = 5$ the model uses one predictor;
   with $m_{\text{stop}} = 500$ it might eventually pick up `x2` as well, fitting
   noise. Choosing $m_{\text{stop}}$ by cross-validated risk
   ([](model_selection.ipynb)) is what stops before the noise-fitting begins.

3. **The procedure is greedy and frozen.** Once `x1`'s coefficient is
   incremented, it is never re-estimated — only added to. This is the
   forward-stagewise property that distinguishes boosting from forward
   *stepwise* regression ([](from_glm_to_boosting.ipynb) §1.2).

## See also

- **The math**: [](functional_gradient_boosting.ipynb) gives the algorithm in full
  generality; [](families.ipynb) tabulates the pseudo-residual for every loss.
- **The code**: [](../usage/quickstart) runs the same workflow at $n=400$
  with a smooth term added; [](../usage/glmboost) does it at $p=20$ with
  automatic variable selection.
- **The engine**: [](../implementation/engine) traces the first
  iteration through the Rust loop — which buffer holds the pseudo-residual,
  where the cached factorization is reused, how the SSE is computed without
  forming the $n$-vector $B_j \beta_j$.

```{bibliography}
:filter: docname in docnames
```